# Galaxy cluster weak-lensing mass with DP2 photo-z and CLMM

For the Rubin Science Platform at data.lsst.cloud. <br>
Data Release: Data Preview 2 <br>
Container Size: large <br>
LSST Science Pipelines version: r30 (same as DP2 tutorials 304.3 and 310.1) <br>
Packages: `lsst.rsp`, `lsdb`, `clmm` (≥ 1.18), a CLMM theory backend (`pyccl`)

**Learning objective:** Select background galaxies behind the galaxy cluster PSZ2 G309.43-72.86 with
DP2 photometric redshifts (photo-z), measure their reduced tangential shear profile, and fit the
cluster mass with CLMM.

**Credit:** Object catalog access and galaxy selection follow the Rubin DP2 tutorial
[304.3 "Galaxy cluster weak lensing with HSM"](https://dp2.lsst.io/tutorials/notebook/304/notebook-304-3.html);
photo-z access follows
[310.1 "Photo-z access and visualization"](https://dp2.lsst.io/tutorials/notebook/310/notebook-310-1.html).
This notebook extends `DP2_cluster_mass_fit_color_selection.ipynb` (in this folder), which selected
sources by color and assumed a redshift distribution.

## Overview

PSZ2 G309.43-72.86 is a rich galaxy cluster at redshift $z=0.35$ in DP2. In the previous notebook we
selected background galaxies by color and *assumed* their redshift distribution. Here we use the DP2
photo-z instead: a galaxy is a lensing source if its photo-z is larger than $z_{\rm cluster} + 0.1$,
and the photo-z of the sources also give the lensing efficiency used in the mass model.

1. **Setup** – make CLMM importable on the RSP.
2. **Load the data** (run only once): the DP2 Object catalog (TAP) and the photo-z (LSDB), matched by
   `objectId`.
3. **Define the analysis parameters** in one place.
4. **Select galaxies** (extended, well-measured objects).
5. **Select background galaxies with photo-z.**
6. **Apply shape-quality cuts.**
7. **Convert HSM ellipticities into shear estimates.**
8. **Compute tangential and cross shear** with CLMM.
9. **Build the radial shear profile** in physical units (Mpc).
10. **Compute the lensing efficiency** from the source photo-z.
11. **Fit an NFW halo mass** with CLMM.
12. **Plot the result.**
13. **Caveats and exercises.**

**Tip:** the catalogs from Section 2 are never modified afterwards. To try other choices (for example
another photo-z estimator, or the radial range), change the parameters in Section 3 and rerun from
Section 3 onwards, without loading the data again.

## 1. Setup

### 1.1 One-time installation (only the first time)

CLMM is not part of the LSST Science Pipelines, so we use the latest CLMM code from GitHub.
Open a terminal on the RSP, go to any directory you like, and clone CLMM:

```bash
git clone https://github.com/LSSTDESC/CLMM.git
```

To get newer CLMM code later, run `git pull` inside the `CLMM` folder.

`import clmm` also needs `qp` (used by CLMM for photo-z PDFs; imported even though this notebook
does not use photo-z) and a theory backend (we use CCL). If they are not available in your
RSP environment, uncomment and run the cell below **once**, then restart the kernel
(Kernel → Restart Kernel).

In [ ]:
# %pip install --user qp-prob pyccl

### 1.2 Make CLMM importable

Jupyter runs a notebook from the notebook's own folder, and the CLMM clone's folder needs to be provided. We put it at the front of the Python path, so `import clmm` uses this
GitHub version instead of any older installed version. We also ask CLMM to use the CCL backend.

In [ ]:
import os
import sys

CLMM_PATH = os.path.abspath("./CLMM")  # The clone's top folder, which contains the "clmm" package
sys.path.insert(0, CLMM_PATH)
os.environ["CLMM_MODELING_BACKEND"] = "ccl"  # Must be set before importing clmm

import clmm

print("CLMM version:", clmm.__version__)
print("CLMM loaded from:", clmm.__file__)

### 1.3 Import packages

General packages (numpy, scipy, matplotlib, astropy), the RSP data-access packages (`lsst.rsp` for
TAP and `lsdb` for the photo-z) and the CLMM modules we need.

In [ ]:
import lsdb
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.table import Table, join
from lsst.rsp import RSPDiscovery
from scipy.integrate import quad

import clmm.dataops as da
from clmm.support.sampler import fitters
from clmm.utils import (
    compute_beta_s,
    compute_beta_s_mean_from_distribution,
    compute_beta_s_square_mean_from_distribution,
    make_bins,
    redshift_distributions,
)

Make the axis labels and tick labels in all figures 1.7 times larger than the matplotlib default.

In [ ]:
label_fontsize = 1.7 * plt.rcParams["font.size"]
plt.rcParams.update(
    {
        "axes.labelsize": label_fontsize,
        "xtick.labelsize": label_fontsize,
        "ytick.labelsize": label_fontsize,
    }
)

## 2. Load the data

This section is run **once**. It depends only on the cluster position (we use the Brightest Cluster
Galaxy, BCG, as the center), the band used for galaxy shapes, and the search radius.

In [ ]:
# Cluster PSZ2 G309.43-72.86 (BCG position used as the center)
cluster_ra = 10.2082  # [deg]
cluster_dec = -44.1307  # [deg]

band_shape = "i"  # Band used for galaxy shapes
radius_deg = 0.4  # Cone-search radius around the BCG [deg]

### 2.1 Object catalog (TAP)

Get the TAP service, exactly as in tutorial 304.3.

In [ ]:
discovery = RSPDiscovery("dp2")
service = discovery.get_tap_client()

Query the `dp2.Object` table for objects within `radius_deg` of the BCG. We ask for the HSM
Regaussianization shapes in the `band_shape` band, the riz CModel photometry, and quality flags.
The signal-to-noise ratio (SNR) is computed in the query because it is not a column in the schema.

In [ ]:
query = (
    "SELECT objectId, coord_ra, coord_dec, refExtendedness, "
    f"{band_shape}_blendedness, "
    f"{band_shape}_hsmShapeRegauss_e1, "
    f"{band_shape}_hsmShapeRegauss_e2, "
    f"{band_shape}_hsmShapeRegauss_sigma, "
    f"{band_shape}_hsmShapeRegauss_flag, "
    "r_cModelMag, i_cModelMag, z_cModelMag, "
    "r_cModelFlux/r_cModelFluxErr AS r_cModel_SNR, "
    "i_cModelFlux/i_cModelFluxErr AS i_cModel_SNR, "
    "z_cModelFlux/z_cModelFluxErr AS z_cModel_SNR, "
    "r_cModel_flag, i_cModel_flag, z_cModel_flag "
    "FROM dp2.Object "
    "WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), "
    f"CIRCLE('ICRS', {cluster_ra}, {cluster_dec}, {radius_deg})) = 1 "
)

Run the query, check that it completed, and convert the result into an astropy table.

In [ ]:
job = service.submit_job(query)
job.run()
job.wait(phases=["COMPLETED", "ERROR"])
print("Job phase is", job.phase)
if job.phase == "ERROR":
    job.raise_if_error()
assert job.phase == "COMPLETED"
obj_cat = job.fetch_result().to_table()
print("Number of objects is", len(obj_cat))

### 2.2 Photo-z catalog (LSDB)

The DP2 photo-z are provided as LSDB-formatted files on the RSP (not as a TAP table), described in
"Photometric Redshifts for Data Preview 2" (RTN-124). Six estimators are available: `bpz`, `dnf`,
`fzboost`, `gpz`, `knn` and `tpz`. For each, we load the point estimate `z_best` (the minimal-risk
estimate of Tanaka et al. 2018) and the lower and upper ends of the 68% credible interval.

We load all six estimators at once (the cone is small), so that changing the estimator in Section 3
does not require loading the data again. Only the objects within `radius_deg` of the BCG are read.

Note: the DP2 photo-z are only lightly validated and have no quality flags yet; treat them as
provisional (see tutorial 310.2).

In [ ]:
pz_codes = ["bpz", "dnf", "fzboost", "gpz", "knn", "tpz"]
pz_columns = []
for pz_code_i in pz_codes:
    pz_columns += [
        f"{pz_code_i}_z_best",
        f"{pz_code_i}_z_err68_low",
        f"{pz_code_i}_z_err68_high",
    ]

pz_lsdb = lsdb.open_catalog(
    "/rubin/lsdb_data/dp2/object_photoz",
    search_filter=lsdb.ConeSearch(
        ra=cluster_ra, dec=cluster_dec, radius_arcsec=radius_deg * 3600.0
    ),
    columns=["objectId", *pz_columns],
)
pz_df = pz_lsdb.compute()
print("Number of objects with photo-z is", len(pz_df))

### 2.3 Match the photo-z to the Object catalog

Both catalogs share the `objectId` column, so we match them by ID (no positional cross-match is
needed). First convert the photo-z table into an astropy table with plain numpy columns (missing
values become NaN), then keep the objects present in both catalogs (an "inner" join).

In [ ]:
pz_table = Table()
pz_table["objectId"] = pz_df["objectId"].to_numpy(dtype=np.int64)
for col in pz_columns:
    pz_table[col] = pz_df[col].to_numpy(dtype=float, na_value=np.nan)

obj_pz_cat = join(obj_cat, pz_table, keys="objectId", join_type="inner")
print("Number of objects in the Object catalog:", len(obj_cat))
print("Number of objects with a photo-z match: ", len(obj_pz_cat))

## 3. Define the analysis parameters

All analysis choices are collected here. **After changing any of them, rerun from this cell
onwards**; there is no need to load the data again.

* **Cluster redshift:** $z = 0.35$.
* **Photo-z:** the FlexZBoost estimator (`fzboost`), which RTN-124 found to perform best on DP2 (see
  tutorial 310.2). Sources must have photo-z $> z_{\rm cluster} + 0.1$.
* **Selection:** the same galaxy and shape cuts as tutorial 304.3 (no red-sequence color cut).
* **Shear calibration:** DP2 does not provide multiplicative ($m$) or additive ($c$) shear biases
  for the HSM shapes, so we set $m = c = 0$. We assume a per-component shape dispersion
  $e_{\rm rms} = 0.365$.
* **Radial bins:** the same bins as tutorial 304.3, which uses 5 logarithmic bins from 4 to 24 arcmin.
  At $z = 0.35$ in our cosmology, 1 arcmin = 0.2964 Mpc, so this is 1.186 to 7.113 Mpc.
* **Model:** NFW profile with $M_{200c}$ (mass inside the radius where the mean density is 200 times
  the critical density) and a fixed concentration $c_{200c} = 4$.
* **Cosmology:** flat $\Lambda$CDM with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$.

In [ ]:
cluster_z = 0.35

# Photo-z
pz_code = "fzboost"  # Photo-z estimator: one of pz_codes (Section 2.2)
dz_min = 0.1  # Sources must have photo-z > cluster_z + dz_min

# Galaxy selection (same as tutorial 304.3)
band_list = ["r", "i", "z"]
snr_min = 5.0

# Shape-quality cuts (same as tutorial 304.3)
e_max = 2.0
sigma_e_max = 0.4
blendedness_max = 0.42

# Shear calibration
e_rms = 0.365  # Assumed per-component intrinsic shape dispersion
m_bias = 0.0  # Multiplicative shear bias
c1_bias = 0.0  # Additive shear bias (component 1)
c2_bias = 0.0  # Additive shear bias (component 2)

# Radial profile: logarithmic bins in projected physical distance [Mpc]
# Default: the tutorial 304.3 bins, 4 to 24 arcmin (= 1.186 to 7.113 Mpc at z = 0.35) in 5 bins
r_min_mpc = 1.2
r_max_mpc = 7.1
n_bins = 5

# Halo model
concentration = 4.0
massdef = "critical"
delta_mdef = 200

# Lensing efficiency
z_inf = 10.0  # "Infinite" redshift used to normalize the lensing efficiency

# Cosmology
cosmo = clmm.Cosmology(H0=70.0, Omega_dm0=0.3 - 0.045, Omega_b0=0.045, Omega_k0=0.0)

## 4. Select galaxies

Keep objects that are extended (`refExtendedness == 1`, i.e. not point-like stars), have no
CModel failure flag, and have SNR > 5 in all of r, i and z.

In [ ]:
sel = obj_pz_cat["refExtendedness"] == 1
for band in band_list:
    sel &= obj_pz_cat[f"{band}_cModel_flag"] == 0
    sel &= obj_pz_cat[f"{band}_cModel_SNR"] > snr_min
gal_cat = obj_pz_cat[sel]  # New table: obj_pz_cat itself is never modified
print("Number of galaxies is", len(gal_cat))

## 5. Select background galaxies with photo-z

A galaxy can only be lensed by the cluster if it is behind the cluster. We keep galaxies whose photo-z
point estimate is larger than $z_{\rm cluster} + 0.1$. The margin of 0.1 reduces contamination by
cluster members and foreground galaxies whose photo-z scatter slightly above the cluster redshift.
Galaxies without a valid photo-z are not selected.

In [ ]:
z_col = f"{pz_code}_z_best"
z_phot = np.asarray(gal_cat[z_col], dtype=float)
z_src_min = cluster_z + dz_min

sel_bg = z_phot > z_src_min
print(f"Photo-z estimator: {pz_code}")
print("Number of galaxies with a valid photo-z is", np.sum(np.isfinite(z_phot)))
print(f"Number of background galaxies (z > {z_src_min:.2f}) is", np.sum(sel_bg))

Plot the photo-z distribution of all galaxies and of the selected background galaxies. The orange line
marks the cluster redshift and the dashed line the selection cut.

In [ ]:
z_bins = np.linspace(0.0, 3.0, 61)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(z_phot[np.isfinite(z_phot)], bins=z_bins, histtype="step", color="k", label="All galaxies")
ax.hist(z_phot[sel_bg], bins=z_bins, histtype="stepfilled", alpha=0.4, label="Background galaxies")
ax.axvline(cluster_z, color="orange", label="Cluster redshift")
ax.axvline(z_src_min, color="grey", ls="--", label=r"$z_{\rm cluster} + 0.1$")
ax.set_xlabel(f"Photo-z ({pz_code} z_best)")
ax.set_ylabel("Number of galaxies")
ax.legend()
plt.show()

## 6. Apply shape-quality cuts

Following tutorial 304.3 (and Mandelbaum et al. 2018), keep background galaxies with:
* total ellipticity $\sqrt{e_1^2 + e_2^2} < 2$ (removes unphysical shapes),
* shape uncertainty `hsmShapeRegauss_sigma` ≤ 0.4,
* no HSM failure flag,
* blendedness < 0.42 (flux not strongly affected by neighbors).

In [ ]:
e1_col = f"{band_shape}_hsmShapeRegauss_e1"
e2_col = f"{band_shape}_hsmShapeRegauss_e2"

sel_shape = np.sqrt(gal_cat[e1_col] ** 2 + gal_cat[e2_col] ** 2) < e_max
sel_shape &= gal_cat[f"{band_shape}_hsmShapeRegauss_sigma"] <= sigma_e_max
sel_shape &= gal_cat[f"{band_shape}_hsmShapeRegauss_flag"] == 0
sel_shape &= gal_cat[f"{band_shape}_blendedness"] < blendedness_max

src_cat = gal_cat[sel_bg & sel_shape]
print("Number of source galaxies is", len(src_cat))

## 7. From ellipticity to shear

The HSM Regaussianization ellipticity is a *distortion*,
$e = (a^2 - b^2)/(a^2 + b^2)$ for an ellipse with axes $a$ and $b$. For a population of galaxies, the
mean distortion responds to a shear $g$ as $\langle e \rangle \approx 2\mathcal{R}\, g$, where the
*responsivity* is

$$
\mathcal{R} = 1 - e_{\rm rms}^2 ,
$$

and $e_{\rm rms}$ is the per-component intrinsic shape dispersion. Including multiplicative ($m$) and
additive ($c$) biases, the shear estimate of each galaxy is (Mandelbaum et al. 2018):

$$
g_{1,2} = \frac{1}{1+m}\left(\frac{e_{1,2}}{2\mathcal{R}} - c_{1,2}\right).
$$

Here $e_{\rm rms} = 0.365$ and $m = c = 0$ (see Section 3).

In [ ]:
responsivity = 1.0 - e_rms**2
print(f"Responsivity R = {responsivity:.3f}")

e1 = np.asarray(src_cat[e1_col], dtype=float)
e2 = np.asarray(src_cat[e2_col], dtype=float)
g1 = (e1 / (2.0 * responsivity) - c1_bias) / (1.0 + m_bias)
g2 = (e2 / (2.0 * responsivity) - c2_bias) / (1.0 + m_bias)

src_ra = np.asarray(src_cat["coord_ra"], dtype=float)
src_dec = np.asarray(src_cat["coord_dec"], dtype=float)
src_z = np.asarray(src_cat[z_col], dtype=float)

## 8. Tangential and cross shear with CLMM

Lensing stretches background galaxies *tangentially* around the cluster. For each source, CLMM
computes the angular separation $\theta$ from the cluster center, the tangential shear
$g_t = -(g_1\cos 2\varphi + g_2 \sin 2\varphi)$ and the cross shear
$g_\times = g_1 \sin 2\varphi - g_2\cos 2\varphi$, where $\varphi$ is the position angle of the source.

DP2 shapes use the same convention as CLMM's `coordinate_system="euclidean"`
($\varphi$ measured from West, counterclockwise), so no sign change is needed.
`geometry="curve"` uses exact spherical geometry.

In [ ]:
theta, gt, gx = da.compute_tangential_and_cross_components(
    ra_lens=cluster_ra,
    dec_lens=cluster_dec,
    ra_source=src_ra,
    dec_source=src_dec,
    shear1=g1,
    shear2=g2,
    coordinate_system="euclidean",
    geometry="curve",
)

**Check:** compare with the formula used in tutorial 304.3 (astropy position angle + $\pi/2$).
The two must agree.

In [ ]:
center = SkyCoord(cluster_ra, cluster_dec, frame="icrs", unit="deg")
sources = SkyCoord(src_ra, src_dec, frame="icrs", unit="deg")
phi = center.position_angle(sources).rad + np.pi / 2.0
gt_tutorial = -g1 * np.cos(2.0 * phi) - g2 * np.sin(2.0 * phi)
gx_tutorial = g1 * np.sin(2.0 * phi) - g2 * np.cos(2.0 * phi)

print("Tangential shear agrees:", np.allclose(gt, gt_tutorial))
print("Cross shear agrees:", np.allclose(gx, gx_tutorial))

## 9. Radial shear profile

Average $g_t$ and $g_\times$ in logarithmic bins of projected physical distance $R$ (Mpc), using the
cluster redshift and our cosmology to convert angles into distances. The error bar in each bin is the
standard error of the mean ($\sigma/\sqrt{N}$), dominated by galaxy shape noise.

By default we use the bins of tutorial 304.3 (5 logarithmic bins from 4 to 24 arcmin, i.e. 1.186 to
7.113 Mpc). The center is skipped because there the source sample is most contaminated by cluster
members and the data are noisy. The next cell prints the bin edges in both Mpc and arcmin, so you
can check the conversion.

In [ ]:
bin_edges = make_bins(r_min_mpc, r_max_mpc, n_bins, method="evenlog10width")
bin_edges_arcmin = clmm.utils.convert_units(bin_edges, "Mpc", "arcmin", cluster_z, cosmo)
print("Bin edges [Mpc]:   ", np.round(bin_edges, 3))
print("Bin edges [arcmin]:", np.round(bin_edges_arcmin, 2))

Compute the binned profile.

In [ ]:
profile = da.make_radial_profile(
    [gt, gx],
    angsep=theta,
    angsep_units="radians",
    bin_units="Mpc",
    bins=bin_edges,
    cosmo=cosmo,
    z_lens=cluster_z,
    coordinate_system="euclidean",
)
# make_radial_profile names the components p_0, p_1; give them clearer names.
profile.rename_columns(["p_0", "p_0_err", "p_1", "p_1_err"], ["gt", "gt_err", "gx", "gx_err"])
profile["radius", "gt", "gt_err", "gx", "gx_err", "n_src"]

Plot the profile. A lensing signal shows up as positive $g_t$ that decreases with radius, while
$g_\times$ should be consistent with zero.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.errorbar(profile["radius"], profile["gt"], profile["gt_err"], fmt="o", label=r"$g_t$")
ax.errorbar(
    profile["radius"] * 1.03, profile["gx"], profile["gx_err"], fmt="x", label=r"$g_\times$"
)
ax.axhline(0, color="k", ls=":")
ax.set_xscale("log")
ax.set_xlabel("R [Mpc]")
ax.set_ylabel("Reduced shear")
ax.legend()
plt.show()

## 10. Compute the lensing efficiency from the photo-z

The lensing signal depends on how far each source is behind the cluster, through the lensing
efficiency

$$
\beta_s(z_s) = \frac{D_{LS}}{D_S}\,\frac{D_\infty}{D_{L\infty}},
$$

where $D_{LS}$ and $D_S$ are angular-diameter distances from the lens and from us to the source.
With photo-z, we compute $\beta_s$ for every source and average $\beta_s$ and $\beta_s^2$ over the
source sample. These two numbers are all the model needs to know about the sources.

In [ ]:
beta_s = compute_beta_s(src_z, cluster_z, z_inf, cosmo)
beta_s_mean = np.mean(beta_s)
beta_s_square_mean = np.mean(beta_s**2)
print(f"<beta_s>   = {beta_s_mean:.3f}")
print(f"<beta_s^2> = {beta_s_square_mean:.3f}")

For comparison, the previous notebook assumed the Chang et al. (2013) redshift distribution for the
sources. Compute its $\langle\beta_s\rangle$ and $\langle\beta_s^2\rangle$ with the same lower redshift
cut, and compare the two distributions.

In [ ]:
chang_kwargs = {
    "z_cl": cluster_z,
    "z_inf": z_inf,
    "cosmo": cosmo,
    "zmin": z_src_min,
    "zmax": 5.0,
    "z_distrib_func": redshift_distributions.chang2013,
}
print(f"Chang+2013: <beta_s>   = {compute_beta_s_mean_from_distribution(**chang_kwargs):.3f}")
print(f"Chang+2013: <beta_s^2> = {compute_beta_s_square_mean_from_distribution(**chang_kwargs):.3f}")

z_grid = np.linspace(z_src_min, 3.0, 300)
chang_norm = quad(redshift_distributions.chang2013, z_src_min, 5.0)[0]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(src_z, bins=z_bins, density=True, histtype="step", color="k", label="Sources (photo-z)")
ax.plot(z_grid, redshift_distributions.chang2013(z_grid) / chang_norm, label="Chang+2013")
ax.axvline(cluster_z, color="orange", label="Cluster redshift")
ax.set_xlabel("Source redshift z")
ax.set_ylabel("N(z) (normalized)")
ax.legend()
plt.show()

## 11. Fit the halo mass

The model is the reduced tangential shear of an NFW halo with mass $M_{200c}$ and fixed concentration.
With `z_src_info="beta"` and `approx="type1"`, CLMM uses the approximation of Applegate et al. (2014):

$$
g_t(R) \approx \frac{\langle\beta_s\rangle\,\gamma_\infty(R)}
{1 - \frac{\langle\beta_s^2\rangle}{\langle\beta_s\rangle}\kappa_\infty(R)} ,
$$

where $\gamma_\infty$ and $\kappa_\infty$ are the shear and convergence for a source at $z_\infty$.
Here $\langle\beta_s\rangle$ and $\langle\beta_s^2\rangle$ come from the source photo-z (Section 10).
We fit $\log_{10} M_{200c}$ with `scipy.optimize.curve_fit` (through CLMM's `fitters`).

In [ ]:
def gt_model(radius, logm):
    return clmm.compute_reduced_tangential_shear(
        r_proj=radius,
        mdelta=10**logm,
        cdelta=concentration,
        z_cluster=cluster_z,
        z_src=(beta_s_mean, beta_s_square_mean),
        cosmo=cosmo,
        delta_mdef=delta_mdef,
        massdef=massdef,
        halo_profile_model="nfw",
        z_src_info="beta",
        approx="type1",
    )

In [ ]:
popt, pcov = fitters["curve_fit"](
    gt_model,
    profile["radius"],
    profile["gt"],
    profile["gt_err"],
    bounds=[12.0, 16.0],
)
logm_fit, logm_err = popt[0], np.sqrt(pcov[0][0])
m_fit = 10**logm_fit
m_err = m_fit * np.log(10) * logm_err

print(f"log10(M200c / Msun) = {logm_fit:.2f} +/- {logm_err:.2f}")
print(f"M200c = ({m_fit / 1e14:.2f} +/- {m_err / 1e14:.2f}) x 10^14 Msun")

## 12. Plot the result

Data points with the best-fit NFW model (orange line). The fit is done in $\log_{10} M_{200c}$, and
$\sigma$ is the 1$\sigma$ statistical uncertainty of the fitted $\log_{10} M_{200c}$ (from the
`curve_fit` covariance). The shaded band shows the model for $\log_{10} M_{200c} \pm \sigma$, i.e. for
masses between $M_{200c} \times 10^{-\sigma}$ and $M_{200c} \times 10^{+\sigma}$ (concentration fixed).

Note: the $\pm$ on $M_{200c}$ in the legend is the first-order conversion
$\sigma_M = M_{200c} \ln(10)\, \sigma$, so it is symmetric, while the band is not.

In [ ]:
r_model = np.logspace(np.log10(r_min_mpc), np.log10(r_max_mpc), 100)

fig, ax = plt.subplots(figsize=(9, 6))
ax.errorbar(profile["radius"], profile["gt"], profile["gt_err"], fmt="o", c="k", label=r"$g_t$ data")
ax.errorbar(
    profile["radius"] * 1.03,
    profile["gx"],
    profile["gx_err"],
    fmt="x",
    c="grey",
    label=r"$g_\times$ data",
)
ax.plot(
    r_model,
    gt_model(r_model, logm_fit),
    "C1",
    label=rf"NFW fit: $M_{{200c}} = ({m_fit/1e14:.1f} \pm {m_err/1e14:.1f})\times10^{{14}}\,M_\odot$",
)
ax.fill_between(
    r_model,
    gt_model(r_model, logm_fit - logm_err),
    gt_model(r_model, logm_fit + logm_err),
    color="C1",
    alpha=0.3,
)
ax.axhline(0, color="k", ls=":")
ax.set_xscale("log")
ax.set_xlabel("R [Mpc]")
ax.set_ylabel("Reduced shear")
ax.legend(loc="upper right", fontsize=label_fontsize)
plt.show()

## 13. Caveats and exercises

This is a simple demonstration. The main simplifications are:

* **Photo-z errors.** Photo-z scatter and outliers put some cluster members and foreground galaxies
  above the redshift cut. They carry no lensing signal (or a weaker one), so they dilute $g_t$ and
  bias the mass **low**. Errors in the source photo-z also bias $\langle\beta_s\rangle$, and hence the
  mass. The DP2 photo-z are provisional and only lightly validated.
* **Point estimates.** We use one point estimate (`z_best`) per galaxy, not the full photo-z
  probability distribution.
* **Shear calibration.** We set $m = c = 0$ and fixed $e_{\rm rms} = 0.365$; galaxies are not weighted.
* **Model.** Fixed concentration, BCG as the center (no miscentering), and no 2-halo term. The
  default bins (tutorial 304.3) reach 7.1 Mpc, where the 2-halo term from surrounding large-scale
  structure is no longer negligible.

**Exercises** (each needs only Section 3 and later to be rerun)

1. Use the BPZ photo-z (`pz_code = "bpz"`) and compare the source sample and the mass.
2. Use a stricter source cut that also requires the lower end of the 68% photo-z interval to be
   behind the cluster: `sel_bg = (z_phot > z_src_min) & (gal_cat[f"{pz_code}_z_err68_low"] > z_src_min)`.
3. Change `dz_min`, or the radial range (`r_min_mpc`, `r_max_mpc`).
4. Instead of $\langle\beta_s\rangle$ and $\langle\beta_s^2\rangle$, model each radial bin with the
   individual source redshifts in that bin (`z_src_info="discrete"`; see the "z,R" model in the CLMM
   HSC example).
5. Compare the mass with the one from `DP2_cluster_mass_fit_color_selection.ipynb`.